# 🏠 House Price Prediction – Linear Regression Model

**Dataset:** 500 houses | 9 features  
**Algorithm:** Linear Regression (Scikit-learn)  
**Workflow:** Load → Clean → Train → Test → Predict → Evaluate

---
### Table of Contents
1. [Import Libraries & Load Data](#1)
2. [Data Preprocessing](#2)
3. [Exploratory Analysis](#3)
4. [Model Training](#4)
5. [Predictions & Evaluation](#5)
6. [Visualizations](#6)


## 1. Import Libraries & Load Data <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 110})

# Load dataset
df = pd.read_csv('house_price.csv')

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()


In [ ]:
# Dataset overview
print("Column Info:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())


## 2. Data Preprocessing <a id='2'></a>

**Steps:**
- Missing values fill karo (median se)
- Features aur Target alag karo
- Train/Test split (80/20)
- Feature Scaling (StandardScaler)


In [ ]:
# Fill missing values with median
df['area_sqft']              = df['area_sqft'].fillna(df['area_sqft'].median())
df['age_of_house']           = df['age_of_house'].fillna(df['age_of_house'].median())
df['distance_from_city_km']  = df['distance_from_city_km'].fillna(df['distance_from_city_km'].median())

print("Missing values after cleaning:", df.isnull().sum().sum())
print("All clean!")


In [ ]:
# Features (X) aur Target (y) alag karo
X = df.drop('price', axis=1)   # Input features
y = df['price']                 # Target variable

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("\nFeature columns:", X.columns.tolist())


In [ ]:
# Train/Test Split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training data:  {X_train.shape[0]} rows")
print(f"Testing data:   {X_test.shape[0]} rows")


In [ ]:
# Feature Scaling — sab features same scale pe laao
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done!")
print("Mean of scaled training data:", X_train_scaled.mean().round(4))


## 3. Exploratory Analysis <a id='3'></a>

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('House Price Distribution', fontsize=13, fontweight='bold')

axes[0].hist(df['price']/100000, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price (in Lakhs)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['price'].mean()/100000, color='red', linestyle='--',
                label=f"Mean: {df['price'].mean()/100000:.1f}L")
axes[0].legend()

axes[1].scatter(df['area_sqft'], df['price']/100000, alpha=0.4, color='#e74c3c', s=25)
axes[1].set_title('Area vs Price')
axes[1].set_xlabel('Area (sqft)')
axes[1].set_ylabel('Price (Lakhs)')

plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(9, 6))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Model Training <a id='4'></a>

**Algorithm: Linear Regression**  
Formula: `Price = w1*area + w2*bedrooms + w3*bathrooms + ... + bias`


In [ ]:
# Model banao aur train karo
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Model training complete!")
print(f"\nIntercept (bias): {model.intercept_:,.0f}")
print("\nFeature Coefficients:")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))


## 5. Predictions & Evaluation <a id='5'></a>

**Metrics:**
- **MAE** — Average error in rupees
- **RMSE** — Root mean squared error
- **R² Score** — Model accuracy (1.0 = perfect)


In [ ]:
# Predictions karo
y_pred = model.predict(X_test_scaled)

# Evaluation metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("="*45)
print("       MODEL EVALUATION RESULTS")
print("="*45)
print(f"  MAE   (Mean Abs Error) : Rs {mae:>12,.0f}")
print(f"  RMSE  (Root MSE)       : Rs {rmse:>12,.0f}")
print(f"  R2 Score (Accuracy)    :    {r2:>10.4f}")
print("="*45)
print(f"  Model Accuracy         :    {r2*100:.2f}%")
print("="*45)


In [ ]:
# Sample Predictions vs Actual
results = pd.DataFrame({
    'Actual Price':    y_test.values[:10],
    'Predicted Price': y_pred[:10].astype(int),
})
results['Difference'] = results['Actual Price'] - results['Predicted Price']
results['Error %'] = (abs(results['Difference']) / results['Actual Price'] * 100).round(2)
print("Sample Predictions (first 10):")
print(results.to_string(index=False))


In [ ]:
# Custom Prediction — apna ghar ka price predict karo!
new_house = pd.DataFrame([{
    'area_sqft': 1800,
    'bedrooms': 3,
    'bathrooms': 2,
    'age_of_house': 5,
    'distance_from_city_km': 10,
    'garage': 1,
    'garden': 1,
    'floors': 2
}])

new_house_scaled = scaler.transform(new_house)
predicted_price  = model.predict(new_house_scaled)[0]

print("="*40)
print("   CUSTOM HOUSE PREDICTION")
print("="*40)
print(f"  Area        : 1800 sqft")
print(f"  Bedrooms    : 3")
print(f"  Bathrooms   : 2")
print(f"  Age         : 5 years")
print(f"  Distance    : 10 km from city")
print(f"  Garage      : Yes")
print(f"  Garden      : Yes")
print(f"  Floors      : 2")
print("="*40)
print(f"  Predicted Price: Rs {predicted_price:,.0f}")
print("="*40)


## 6. Visualizations <a id='6'></a>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Visualizations', fontsize=14, fontweight='bold')

# 1. Actual vs Predicted
axes[0,0].scatter(y_test, y_pred, alpha=0.5, color='#3498db', s=30)
mn, mx = y_test.min(), y_test.max()
axes[0,0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect Prediction')
axes[0,0].set_title('Actual vs Predicted Price', fontsize=12)
axes[0,0].set_xlabel('Actual Price')
axes[0,0].set_ylabel('Predicted Price')
axes[0,0].legend()

# 2. Residuals
residuals = y_test - y_pred
axes[0,1].scatter(y_pred, residuals, alpha=0.5, color='#e74c3c', s=30)
axes[0,1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[0,1].set_title('Residual Plot', fontsize=12)
axes[0,1].set_xlabel('Predicted Price')
axes[0,1].set_ylabel('Residuals (Error)')

# 3. Feature Importance
coef_df_sorted = coef_df.sort_values('Coefficient')
colors = ['#e74c3c' if v < 0 else '#27ae60' for v in coef_df_sorted['Coefficient']]
axes[1,0].barh(coef_df_sorted['Feature'], coef_df_sorted['Coefficient'], color=colors, edgecolor='white')
axes[1,0].axvline(0, color='black', linewidth=1)
axes[1,0].set_title('Feature Importance (Coefficients)', fontsize=12)
axes[1,0].set_xlabel('Coefficient Value')

# 4. Error Distribution
axes[1,1].hist(residuals/1000, bins=30, color='#9b59b6', edgecolor='white', alpha=0.8)
axes[1,1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1,1].set_title('Error Distribution', fontsize=12)
axes[1,1].set_xlabel('Error (in Thousands)')
axes[1,1].set_ylabel('Count')

plt.tight_layout()
plt.show()


## Summary 📊

| Metric | Value |
|--------|-------|
| Algorithm | Linear Regression |
| Training Data | 400 houses (80%) |
| Testing Data | 100 houses (20%) |
| R² Score | ~0.75+ |
| Key Features | Area, Bedrooms, Distance from City |

### Key Learnings
- **Area** sabse zyada price affect karta hai
- **Distance from city** price kam karta hai (negative impact)
- **Garage + Garden** price badhate hain
- Model ~75%+ accuracy se price predict kar sakta hai
